In [1]:
import ee
import pandas as pd

import geemap
ee.Initialize(project='induswater')
ee_object = geemap.shp_to_ee('indus_shapefile/upperindusbd.shp')

# ============================================================
# QA MASKING
# ============================================================
def bitwiseExtract(input, from_bit, to_bit):
    maskSize = ee.Number(1).add(to_bit).subtract(from_bit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(from_bit).bitwiseAnd(mask)


def maskMODISNDVI(image):
    qa_band = image.select('DetailedQA')

    # Bits 0-1: VI quality = 0 (good quality only)
    vi_quality = bitwiseExtract(qa_band, 0, 1).eq(0)

    # Bits 2-5: VI usefulness <= 1 (highest + lower quality only)
    # 2 and above is "decreasing quality" - reject those
    vi_usefulness = bitwiseExtract(qa_band, 2, 5).lte(1)

    # Bit 8: Adjacent cloud
    adjacent_cloud = qa_band.bitwiseAnd(1 << 8).eq(0)

    # Combine QA masks
    mask = vi_quality.And(vi_usefulness).And(adjacent_cloud)

    # Valid range mask (excludes -3000 fill value)
    ndvi = image.select('NDVI')
    valid_range_mask = ndvi.gt(-2000)

    final_mask = mask.And(valid_range_mask)

    # Apply mask and scale to 0-1 range
    return ndvi.updateMask(final_mask).multiply(0.0001) \
               .set('system:time_start', image.get('system:time_start'))


# ============================================================
# LOAD AND MASK DATASET
# ============================================================
ndvi_dataset = ee.ImageCollection('MODIS/061/MOD13Q1') \
    .filterDate('2000-02-18', '2026-01-01') \
    .select(['NDVI', 'DetailedQA'])

ndvi_masked = ndvi_dataset.map(maskMODISNDVI)


# ============================================================
# EXTRACT - iterate by 16-day windows, NOT daily
# ============================================================
def process_image(image):
    """Reduce each 16-day composite image to a single regional mean."""
    regional_stats = image.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=ee_object.geometry(),
        scale=250,
        maxPixels=1e9
    )

    # Get the date from the image metadata
    date = ee.Date(image.get('system:time_start'))

    # Safely get NDVI value - returns None if no valid pixels
    ndvi_val = ee.Number(
        ee.Algorithms.If(
            regional_stats.contains('NDVI'),
            regional_stats.get('NDVI'),
            -9999
        )
    )

    return ee.Feature(None, {
        'date': date.format('YYYY-MM-dd'),
        'NDVI': ndvi_val
    })


# ============================================================
# EXTRACT YEAR BY YEAR (to avoid GEE memory limits)
# ============================================================
ndvi_records = []

for year in range(2000, 2026):
    start_date = f'{year}-01-01' if year!=2000 else '2000-02-18'
    end_date = f'{year + 1}-01-01' if year != 2025 else '2026-01-01'

    # Filter to this year's 16-day composites
    yearly_collection = ndvi_masked.filterDate(start_date, end_date)

    # Map reduceRegion over each 16-day image directly
    features = yearly_collection.map(process_image)
    feature_collection = ee.FeatureCollection(features)

    # Download
    year_data = feature_collection.getInfo()
    year_records = [f['properties'] for f in year_data['features']]
    ndvi_records.extend(year_records)

    print(f'{year} - {len(year_records)} composites extracted')


# ============================================================
# SAVE TO CSV
# ============================================================
df = pd.DataFrame(ndvi_records)
df.to_csv('NDVI_16day_combined.csv', index=False)
print(f'{len(df)} total records saved to NDVI_16day_combined.csv')

2000 - 20 composites extracted
2001 - 23 composites extracted
2002 - 23 composites extracted
2003 - 23 composites extracted
2004 - 23 composites extracted
2005 - 23 composites extracted
2006 - 23 composites extracted
2007 - 23 composites extracted
2008 - 23 composites extracted
2009 - 23 composites extracted
2010 - 23 composites extracted
2011 - 23 composites extracted
2012 - 23 composites extracted
2013 - 23 composites extracted
2014 - 23 composites extracted
2015 - 23 composites extracted
2016 - 23 composites extracted
2017 - 23 composites extracted
2018 - 23 composites extracted
2019 - 23 composites extracted
2020 - 23 composites extracted
2021 - 23 composites extracted
2022 - 23 composites extracted
2023 - 23 composites extracted
2024 - 23 composites extracted
2025 - 23 composites extracted
595 total records saved to NDVI_16day_combined.csv
